In [ ]:
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots

In [ ]:
DISTRIBUTION = "HIERARCHICAL_PAIRS"

In [ ]:
df = pd.read_parquet("data/matryoshka_targeted/results.parquet")

In [ ]:
# Best F1 per benchmark
for bench in df.index.get_level_values("benchmark").unique():
    group = df.loc[bench]
    best = group.loc[group["f1_score"].idxmax()]
    print(f"{bench}: F1={best['f1_score']:.4f} ")

In [ ]:
df.loc[DISTRIBUTION]

## Focus distribution

In [ ]:
sub = df.loc[DISTRIBUTION].reset_index()
sub.head()

## Marginal effects on F1 score

In [ ]:
fig = make_subplots(
    rows=1,
    cols=2,
    subplot_titles=["k", "Nesting Depth (widths)"],
)

for i, col in enumerate(["k", "widths"]):
    fig.add_trace(
        go.Box(
            x=sub[col].astype(str),
            y=sub["f1_score"],
            boxpoints="all",
            jitter=0.3,
            pointpos=0,
            marker_color=px.colors.qualitative.Plotly[i],
        ),
        row=1,
        col=i + 1,
    )

fig.update_layout(
    height=450,
    width=800,
    title_text=f"F1 Score by Hyperparameter ({DISTRIBUTION})",
    showlegend=False,
)
fig.show()

## Interaction heatmaps: k × widths (F1 and MCC)

In [ ]:
fig = make_subplots(
    rows=1,
    cols=2,
    subplot_titles=["F1 Score", "MCC"],
)

for i, metric in enumerate(["f1_score", "mcc"]):
    pivot = sub.pivot_table(values=metric, index="widths", columns="k", aggfunc="mean")
    pivot = pivot.sort_index(ascending=False)
    fig.add_trace(
        go.Heatmap(
            z=pivot.values,
            x=[str(c) for c in pivot.columns],
            y=[str(r) for r in pivot.index],
            colorscale="Viridis",
            showscale=(i == 1),
            text=pivot.values.round(3),
            texttemplate="%{text}",
        ),
        row=1,
        col=i + 1,
    )

fig.update_layout(
    height=300,
    width=900,
    title_text=f"k × widths ({DISTRIBUTION})",
)
fig.update_xaxes(title_text="k")
fig.update_yaxes(title_text="Nesting Depth", col=1)
fig.show()

## F1 vs k line plot (colored by nesting depth)

In [ ]:
melted = sub.melt(
    id_vars=["k", "widths"],
    value_vars=["f1_score", "mcc"],
    var_name="metric",
    value_name="score",
)

fig = px.line(
    melted,
    x="k",
    y="score",
    color="widths",
    facet_col="metric",
    category_orders={
        "widths": ["2-level", "3-level", "4-level"],
        "metric": ["f1_score", "mcc"],
    },
    markers=True,
    labels={
        "score": "Score",
        "k": "k",
        "widths": "Nesting Depth",
        "metric": "Metric",
    },
    title=f"F1 & MCC vs k by nesting depth ({DISTRIBUTION})",
    height=400,
    width=900,
)
fig.show()